# 🛠️ ToolArena demo
## Confusion Attack Tool-Calling Evaluation Benchmark for LLMs

We generate a 25,000-sample benchmark dataset of deliberately ambiguous tool-selection tasks, train six model variants (2 x base, 2 x fine-tuned, 1 x self-distilled, 1 x knowledge-distilled), and evaluate them with objective + subjective metrics, including an LLM-as-Judge and BERTScore on the reasoning chain.


## Setup & Configuration

In [ ]:
# Install all dependencies
!pip install -q transformers==4.44.2 peft==0.12.0 trl==0.11.4 bitsandbytes==0.43.3 \
    datasets==2.20.0 wandb==0.17.9 scikit-learn==1.5.2 \
    bert-score seaborn matplotlib plotly accelerate

In [ ]:
import os, sys, random
import torch

REPO_ROOT = "/content/ToolArena"   # adjust if needed
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

# ── Global constants ──────────────────────────────────────────────────────────
SEED        = 100         # master random seed — every RNG in the project derives from this
DOMAIN      = "bi"        # domain identifier
DEMO_N      = 1000        # demo subset size (tweak here, affects all downstream cells)
FULL_N      = 25_000      # full dataset size
WANDB_PROJECT = "ToolArena"

random.seed(SEED)
torch.manual_seed(SEED)

print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()} | device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"SEED={SEED} | DEMO_N={DEMO_N} | DOMAIN='{DOMAIN}'")

In [ ]:
import wandb

# Try environment variable first, fall back to interactive prompt
_api_key = os.environ.get("WANDB_API_KEY")
if _api_key:
    wandb.login(key=_api_key, relogin=False)
    print("W&B logged in via WANDB_API_KEY environment variable.")
else:
    print("WANDB_API_KEY not set — falling back to interactive login.")
    wandb.login()

## Tool Registry & Confusion Groups


In [ ]:
from core.tool_registry import ToolRegistry

registry = ToolRegistry(domain=DOMAIN)
print(registry.summary())

In [ ]:
import pandas as pd

# Display confusion groups as a DataFrame
group_rows = []
for gid in registry.get_all_group_ids():
    meta  = registry.get_group_meta(gid)
    tools = registry.get_tools_in_group(gid)
    group_rows.append({
        "Group ID":    gid,
        "Tools":       ", ".join(t["name"] for t in tools),
        "Group Size":  len(tools),
    })

df_groups = pd.DataFrame(group_rows)
print(f"\n{len(registry.tools)} tools across {len(registry.confusion_groups)} confusion groups\n")
display(df_groups)

In [ ]:
# Display all 28 tool definitions
tool_rows = []
for t in registry.tools:
    tool_rows.append({
        "Name":             t["name"],
        "Group":            t["confusion_group"],
        "Description":      t["description"][:80] + "...",
    })
display(pd.DataFrame(tool_rows))

## Dataset Generation

### What makes this a confusion-attack dataset?

Each sample's candidate set of 5 tools contains **1 correct tool + 4 distractors**. Distractors are sampled as follows:
- **75%** from the *same confusion group* as the correct tool (high semantic overlap)
- **25%** cross-group (random other tools)

This means the model can never rely on "pick the tool that sounds vaguely relevant". It must understand fine-grained semantic distinctions within each group.

In [ ]:
from core.dataset_generator import DatasetBuilder, load_dataset
from pathlib import Path

FULL_DATASET_PATH = f"datasets/{DOMAIN}/full_dataset.jsonl"

if Path(FULL_DATASET_PATH).exists():
    print(f"Full dataset already exists at {FULL_DATASET_PATH} — skipping generation.")
    full_dataset = load_dataset(FULL_DATASET_PATH)
else:
    builder = DatasetBuilder(domain=DOMAIN, seed=SEED)
    samples = builder.build(n=FULL_N)
    builder.save(samples, output_dir=f"datasets/{DOMAIN}", filename="full_dataset.jsonl")
    full_dataset = [s.to_dict() for s in samples]

print(f"\nFull dataset: {len(full_dataset):,} samples")
print("\nSample record:")
import json; print(json.dumps(full_dataset[0], indent=2))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Full Dataset — Distribution Analysis", fontsize=14, fontweight="bold")

# Difficulty distribution
diff_counts = pd.Series([s["difficulty"] for s in full_dataset]).value_counts().sort_index()
axes[0].bar(["Easy (D1)", "Medium (D2)", "Hard (D3)"], diff_counts.values, color=["#4CAF50","#FF9800","#F44336"])
axes[0].set_title("Difficulty Distribution")
axes[0].set_ylabel("Count")

# Confusion group distribution
group_counts = pd.Series([s["confusion_group"] for s in full_dataset]).value_counts()
axes[1].barh(group_counts.index, group_counts.values, color="#2196F3")
axes[1].set_title("Samples per Confusion Group")
axes[1].set_xlabel("Count")

# Correct tool distribution (top 10)
tool_counts = pd.Series([s["correct_tool"] for s in full_dataset]).value_counts().head(10)
axes[2].barh(tool_counts.index, tool_counts.values, color="#9C27B0")
axes[2].set_title("Top 10 Tools by Sample Count")
axes[2].set_xlabel("Count")

plt.tight_layout()
plt.savefig(f"assets/{DOMAIN}/full_dataset_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → assets/{DOMAIN}/full_dataset_distributions.png")

In [ ]:
# Distractor intra-group ratio verification
intra_counts = []
for s in full_dataset[:500]:   # spot-check first 500
    group    = s["confusion_group"]
    group_tools = {t["name"] for t in registry.get_tools_in_group(group)} - {s["correct_tool"]}
    distractors = s["distractor_tools"]
    intra = sum(1 for d in distractors if d in group_tools)
    intra_counts.append(intra / len(distractors) if distractors else 0)

observed_ratio = sum(intra_counts) / len(intra_counts)
print(f"Target intra-group distractor ratio  : 0.75")
print(f"Observed intra-group distractor ratio : {observed_ratio:.3f}")
print("✓ Confusion-attack design verified." if abs(observed_ratio - 0.75) < 0.05 else "⚠ Ratio mismatch — check ConfusionAttackSampler.")

## Demo Subset Sampling

The demo subset is a stratified sample of `DEMO_N` rows that mirrors the full dataset's distribution across confusion groups and difficulty levels. All training and evaluation in this notebook uses the demo subset to keep runtimes low.

In [ ]:
from core.subset_sampler import StratifiedSampler

DEMO_PATH = f"datasets/{DOMAIN}/demo_subset.jsonl"
sampler   = StratifiedSampler(seed=SEED)

if Path(DEMO_PATH).exists():
    print(f"Demo subset already exists — loading from {DEMO_PATH}")
    from core.dataset_generator import load_dataset as _ld
    demo_subset = _ld(DEMO_PATH)
else:
    demo_subset = sampler.sample(
        full_dataset_path=FULL_DATASET_PATH, n=DEMO_N
    )
    sampler.save(demo_subset, output_path=DEMO_PATH)

print(f"\nDemo subset: {len(demo_subset):,} samples")

In [ ]:
# Train / val / test splits — used by ALL subsequent sections
train_samples, val_samples, test_samples = sampler.compute_splits(demo_subset)
print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

In [ ]:
# Distribution comparison: full dataset vs demo subset
print("=== Full Dataset ===")
print(sampler.stratum_report(full_dataset, label="full dataset"))
print()
print("=== Demo Subset ===")
print(sampler.stratum_report(demo_subset, label="demo subset"))

## Evaluation Framework

In [ ]:
from core.evaluator import EvalRunner
from core.model_predictor import ModelPredictor

# Create ONE EvalRunner — reused for all 6 variants to ensure identical conditions
runner = EvalRunner(
    judge_model_path="microsoft/Phi-3.5-mini-instruct",
    wandb_project=WANDB_PROJECT,
    domain=DOMAIN,
    log_predictions_to_wandb=True,
)
print("EvalRunner ready.")
print(f"Judge model : microsoft/Phi-3.5-mini-instruct")
print(f"Test split  : {len(test_samples)} samples")

In [ ]:
# Baseline 1: base-big (7B, zero-shot)
predictor_base_big = ModelPredictor.from_pretrained(
    model_name_or_path="Qwen/Qwen2.5-7B-Instruct",
    variant_name="base-big",
    load_in_4bit=True,
)
bundle_base_big = runner.run(
    predictor=predictor_base_big,
    test_samples=test_samples,
    variant_name="base-big",
)
print(bundle_base_big.summary())

In [ ]:
# Baseline 2: base-small (0.5B, zero-shot)
predictor_base_small = ModelPredictor.from_pretrained(
    model_name_or_path="Qwen/Qwen2.5-0.5B-Instruct",
    variant_name="base-small",
    load_in_4bit=False,
)
bundle_base_small = runner.run(
    predictor=predictor_base_small,
    test_samples=test_samples,
    variant_name="base-small",
)
print(bundle_base_small.summary())

## Fine-tune Big LLM (`finetuned-big`)

SFT + LoRA on `Qwen2.5-7B-Instruct` in 4-bit. Only the LoRA adapter weights (~0.5% of parameters) are trained. Loss is computed on the assistant turn only. The model learns to produce `{"reasoning": ..., "selected_tool": ...}`.

In [ ]:
from core.finetuner import LoRAFinetuner, LoRAConfig, TrainingConfig

lora_cfg_big = LoRAConfig(r=8, lora_alpha=16)
train_cfg_big = TrainingConfig(
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
)

finetuner_big = LoRAFinetuner(
    base_model_name="Qwen/Qwen2.5-7B-Instruct",
    variant_name="finetuned-big",
    lora_config=lora_cfg_big,
    training_config=train_cfg_big,
    load_in_4bit=True,
    wandb_project=WANDB_PROJECT,
    output_dir="adapters",
)
adapter_path_big = finetuner_big.train(train_samples, val_samples)
print(f"Adapter saved → {adapter_path_big}")

In [ ]:
# Evaluate finetuned-big
predictor_ft_big = ModelPredictor.from_pretrained(
    model_name_or_path="Qwen/Qwen2.5-7B-Instruct",
    variant_name="finetuned-big",
    load_in_4bit=True,
    peft_adapter_path=adapter_path_big,
)
bundle_ft_big = runner.run(
    predictor=predictor_ft_big,
    test_samples=test_samples,
    variant_name="finetuned-big",
)
print(bundle_ft_big.summary())

## Fine-tune Small LLM (`finetuned-small`)

Same SFT + LoRA approach, but on the 0.5B model at full precision. No quantization.

In [ ]:
from core.finetuner import LoRAFinetuner, LoRAConfig, TrainingConfig

lora_cfg_small = LoRAConfig(r=8, lora_alpha=16)
train_cfg_small = TrainingConfig(
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
)

finetuner_small = LoRAFinetuner(
    base_model_name="Qwen/Qwen2.5-0.5B-Instruct",
    variant_name="finetuned-small",
    lora_config=lora_cfg_small,
    training_config=train_cfg_small,
    load_in_4bit=False,
    wandb_project=WANDB_PROJECT,
    output_dir="adapters",
)
adapter_path_small = finetuner_small.train(train_samples, val_samples)
print(f"Adapter saved → {adapter_path_small}")

In [ ]:
# Evaluate finetuned-small
predictor_ft_small = ModelPredictor.from_pretrained(
    model_name_or_path="Qwen/Qwen2.5-0.5B-Instruct",
    variant_name="finetuned-small",
    load_in_4bit=False,
    peft_adapter_path=adapter_path_small,
)
bundle_ft_small = runner.run(
    predictor=predictor_ft_small,
    test_samples=test_samples,
    variant_name="finetuned-small",
)
print(bundle_ft_small.summary())

## Self-Distillation (`self-distilled-base-small`)

The **same model** acts as both teacher and student on each training step:

1. Forward pass with **dropout off** (`model.eval()`) → clean soft targets
2. Forward pass with **dropout on** (`model.train()`) → noisy student logits
3. Loss = `KDLoss(student_logits, teacher_logits, hard_labels)`

This forces the model to produce more calibrated, confident distributions.It is a regularisation technique that can close some of the gap between a small model's default behaviour and its fine-tuned ceiling.

### KD Loss formula

```
L = α · CrossEntropy(student, hard_labels)
  + (1 - α) · T² · KL( softmax(student/T) ‖ softmax(teacher/T) )
```

- **α = 0.5** — equal weight to hard labels and soft targets
- **T = 2.0** — smooths the distribution so near-misses carry signal
- **T²** — rescales KL gradient magnitude to match CE (Hinton et al. 2015)

> **Comparison:** Self-distillation vs `finetuned-small` reveals how much of the gain comes from the *distillation itself* vs. simply seeing more training signal.

In [ ]:
from core.distiller import SelfDistillationTrainer, DistillationConfig

distill_cfg = DistillationConfig(
    alpha=0.5,
    temperature=2.0,
    num_train_epochs=3,
    learning_rate=2e-4,
    gradient_accumulation_steps=8,
    fp16=True,
)

self_kd_trainer = SelfDistillationTrainer(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    variant_name="self-distilled-base-small",
    config=distill_cfg,
    wandb_project=WANDB_PROJECT,
    output_dir="adapters",
)
adapter_path_self_kd = self_kd_trainer.train(train_samples, val_samples)
print(f"Adapter saved → {adapter_path_self_kd}")

In [ ]:
# Evaluate self-distilled-base-small
predictor_self_kd = ModelPredictor.from_pretrained(
    model_name_or_path="Qwen/Qwen2.5-0.5B-Instruct",
    variant_name="self-distilled-base-small",
    load_in_4bit=False,
    peft_adapter_path=adapter_path_self_kd,
)
bundle_self_kd = runner.run(
    predictor=predictor_self_kd,
    test_samples=test_samples,
    variant_name="self-distilled-base-small",
)
print(bundle_self_kd.summary())

## Knowledge Distillation Big→Small (`distilled-base-big-to-base-small`)

A **frozen 7B teacher** provides soft probability distributions over all 28 tools. The **0.5B student** learns to match these distributions, absorbing the teacher's uncertainty and its richer disambiguation knowledge.

| | Self-distillation | Teacher→Student KD |
|---|---|---|
| Teacher | Same model (dropout off) | Separate frozen 7B model |
| Signal | Self-calibration | Transfer of 7B's tool semantics |
| Expected gain | Moderate regularisation | Larger knowledge transfer |


In [ ]:
from core.distiller import KnowledgeDistillationTrainer, DistillationConfig

kd_cfg = DistillationConfig(
    alpha=0.5,
    temperature=2.0,
    num_train_epochs=3,
    learning_rate=2e-4,
    gradient_accumulation_steps=8,
    fp16=True,
)

kd_trainer = KnowledgeDistillationTrainer(
    teacher_model_name="Qwen/Qwen2.5-7B-Instruct",
    student_model_name="Qwen/Qwen2.5-0.5B-Instruct",
    variant_name="distilled-base-big-to-base-small",
    config=kd_cfg,
    wandb_project=WANDB_PROJECT,
    output_dir="adapters",
)
adapter_path_kd = kd_trainer.train(train_samples, val_samples)
print(f"Student adapter saved → {adapter_path_kd}")

In [ ]:
# Evaluate distilled-base-big-to-base-small
predictor_kd = ModelPredictor.from_pretrained(
    model_name_or_path="Qwen/Qwen2.5-0.5B-Instruct",
    variant_name="distilled-base-big-to-base-small",
    load_in_4bit=False,
    peft_adapter_path=adapter_path_kd,
)
bundle_kd = runner.run(
    predictor=predictor_kd,
    test_samples=test_samples,
    variant_name="distilled-base-big-to-base-small",
)
print(bundle_kd.summary())

## Unified Evaluation
All six bundles evaluated on the **same held-out test split**. Collect all `ResultsBundle` objects into a comparison DataFrame.

In [ ]:
# Collect all bundles
all_bundles = [
    bundle_base_big,
    bundle_base_small,
    bundle_ft_big,
    bundle_ft_small,
    bundle_self_kd,
    bundle_kd,
]

# Build comparison DataFrame
comparison_rows = [b.to_flat_dict() for b in all_bundles]
df_results = pd.DataFrame(comparison_rows).set_index("variant_name")

display(df_results[[
    "tool_accuracy", "macro_f1",
    "mean_judge_score", "mean_bertscore_f1", "reasoning_consistency_rate",
    "n_samples", "inference_time_s",
]].round(4))

In [ ]:
# Per-group accuracy comparison
group_ids = registry.get_all_group_ids()
group_acc_data = {}
for b in all_bundles:
    group_acc_data[b.variant_name] = b.per_group_accuracy

df_group_acc = pd.DataFrame(group_acc_data).T
display(df_group_acc.round(3))

## Results & Analysis
Final visualisations saved to `assets/bi/`

In [ ]:
import os
os.makedirs(f"assets/{DOMAIN}", exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("ToolArena — Model Comparison", fontsize=15, fontweight="bold")

variants  = df_results.index.tolist()
colors    = ["#607D8B","#9E9E9E","#1976D2","#42A5F5","#FF7043","#EF5350"]

# Accuracy bar chart
axes[0].barh(variants, df_results["tool_accuracy"], color=colors)
axes[0].set_xlabel("Tool Selection Accuracy")
axes[0].set_title("Accuracy by Variant")
axes[0].set_xlim(0, 1)
for i, v in enumerate(df_results["tool_accuracy"]):
    axes[0].text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)

# Judge score bar chart
axes[1].barh(variants, df_results["mean_judge_score"], color=colors)
axes[1].set_xlabel("Mean LLM-as-Judge Score (1–5)")
axes[1].set_title("Reasoning Quality by Variant")
axes[1].set_xlim(0, 5)
for i, v in enumerate(df_results["mean_judge_score"]):
    axes[1].text(v + 0.05, i, f"{v:.2f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig(f"assets/{DOMAIN}/comparison_bars.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np

# Per-group accuracy heatmap
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    df_group_acc,
    annot=True, fmt=".2f",
    cmap="YlGn", vmin=0, vmax=1,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Per-Confusion-Group Accuracy — All Variants", fontsize=13, fontweight="bold")
ax.set_xlabel("Confusion Group")
ax.set_ylabel("Model Variant")
plt.tight_layout()
plt.savefig(f"assets/{DOMAIN}/group_accuracy_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Confusion matrix for the best-performing variant
best_variant_name = df_results["tool_accuracy"].idxmax()
best_bundle = next(b for b in all_bundles if b.variant_name == best_variant_name)

import numpy as np
cm       = np.array(best_bundle.confusion_matrix)
labels   = best_bundle.cm_labels

fig, ax  = plt.subplots(figsize=(16, 14))
sns.heatmap(
    cm, annot=True, fmt="d",
    xticklabels=labels, yticklabels=labels,
    cmap="Blues", linewidths=0.3,
    ax=ax,
)
ax.set_title(f"Confusion Matrix — {best_variant_name}", fontsize=13, fontweight="bold")
ax.set_xlabel("Predicted Tool")
ax.set_ylabel("True Tool")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(f"assets/{DOMAIN}/confusion_matrix_best.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Best variant: {best_variant_name}  (accuracy={best_bundle.tool_accuracy:.4f})")

### Key Findings

**1. Confusion-attack difficulty** 
Confusion group with the lowest accuracy across all variants: 

**2. Fine-tuning impact** 
SFT + LoRA impact on accuracy vs the base models: 

**3. Distillation vs. fine-tuning** 
Reasoning consistency rate insights: 

**4. Reasoning quality** 
Cases where a model selected the correct tool but received a low judge score:
